## 1. Import Libraries

In [ ]:
# Load libraries
import os, sys #os dung de doc duong dan file, tao xoa thu muc kiem tra file ton tai,sys lay tham so dong lenh vaf dung thoat chuong trinh
from IPython import display # dung cho viec hien hinh anh
import numpy as np
import matplotlib.pyplot as plt # ve do thi 
import pandas as pd # dung su ly du lieu dang bang doc , loc du lieu tu csv excel
import seaborn as sns # ve bieu do nang cao
import joblib # luu va load mo hinh machine learning
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder # chuyen du lieu thanh vecto ,text thanh so, du lieu thu tu
from sklearn.preprocessing import MinMaxScaler, StandardScaler # chuan hoa du lieu tu 0 ->1,Chuẩn hóa dữ liệu theo mean = 0 và std = 1
from sklearn.model_selection import train_test_split # chia va train/ test
from sklearn import model_selection
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import warnings
from sklearn.model_selection import cross_val_score
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams['figure.dpi'] = 100

warnings.filterwarnings("ignore")

## 2. Load Dataset

In [ ]:
data_path = "eda/data/pima-indians-diabetes.csv"

data_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

df_dataset = pd.read_csv(data_path, names=data_names)

In [ ]:
# shape
print(f'+ Shape : {df_dataset.shape}')
# types
print(f'+ Data Types: \n{df_dataset.dtypes}')
# head, tail
print(f'+ Contents: ')
display.display(df_dataset.head(5))
display.display(df_dataset.tail(5))
# info
df_dataset.info()

Ngưỡng sinh lý

In [ ]:
physiological_ranges = {
    "Pregnancies": (0, 15),
    "Glucose": (35, 250),
    "BloodPressure": (40, 180),
    "SkinThickness": (7, 100),
    "Insulin": (15, 900),
    "BMI": (10, 60),
    "DiabetesPedigreeFunction": (0, 2.5),
    "Age": (21, 90)
}

xóa dữ liệu trùng lập

In [ ]:
duplicated = df_dataset.duplicated()

print(f"Số dòng bị trùng: {duplicated.sum()}")

df_clean = df_dataset.drop_duplicates()

print(f"Kích thước sau khi xóa dòng trùng: {df_clean.shape}")

In [ ]:
from xu_ly_du_lieu import xulydulieu

# Áp dụng
df_final = xulydulieu(df_clean,physiological_ranges)

chia dữ liệu 7/3

In [ ]:

X = df_final.drop("Outcome", axis=1)
y = df_final["Outcome"]
seed=1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:

X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=seed, stratify=y_train
)

print("Train sub:", X_train_sub.shape)
print("Validation:", X_valid.shape)

Linear_Discrimnation

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

# Khởi tạo mô hình LDA
# (LDA mặc định hoạt động rất tốt mà không cần tinh chỉnh quá nhiều tham số ban đầu)
model_lda = LinearDiscriminantAnalysis()

# Huấn luyện mô hình
model_lda.fit(X_train, y_train)

# Dự đoán trên tập Test
y_pred_lda = model_lda.predict(X_test)

# Đánh giá độ chính xác và ma trận nhầm lẫn
print("LDA Accuracy:", accuracy_score(y_test, y_pred_lda))

cm_lda = confusion_matrix(y_test, y_pred_lda)
print("Confusion Matrix:\n", cm_lda)

# In các tham số của mô hình
for k, v in model_lda.get_params().items():
    print(f"{k}: {v}")

K-flod

In [ ]:
from sklearn.model_selection import cross_val_score

# Áp dụng K-fold với cv=10 cho mô hình LDA
scores = cross_val_score(model_lda, X_train, y_train, cv=10)

print("Scores từng fold:", scores)
print("Mean Accuracy:", scores.mean())

# In lại các tham số của mô hình để kiểm tra
for k, v in model_lda.get_params().items():
    print(f"{k}: {v}")

In [ ]:
# 1. Dữ liệu thô (Raw)
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

# 2. Xử lý chuẩn hóa Min/Max
from sklearn.preprocessing import MinMaxScaler
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# 3. Xử lý chuẩn hóa Standard (Z-score)
from sklearn.preprocessing import StandardScaler
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

print("--- 1. ĐÁNH GIÁ LDA TRÊN 3 LOẠI DỮ LIỆU (BASELINE) ---")

# Khởi tạo mô hình LDA
lda_eval = LinearDiscriminantAnalysis()

# 1. Chạy trên dữ liệu Thô (Raw)
lda_eval.fit(X_train_raw, y_train)
acc_raw_lda = accuracy_score(y_test, lda_eval.predict(X_test_raw))

# 2. Chạy trên dữ liệu Min/Max
lda_eval.fit(X_train_minmax, y_train)
acc_minmax_lda = accuracy_score(y_test, lda_eval.predict(X_test_minmax))

# 3. Chạy trên dữ liệu Standard
lda_eval.fit(X_train_std, y_train)
acc_std_lda = accuracy_score(y_test, lda_eval.predict(X_test_std))

# --- TÓM TẮT VÀ TRÌNH DIỄN KẾT QUẢ ---
results_lda = pd.DataFrame({
    'Loại dữ liệu': ['Dữ liệu Thô (Raw)', 'Chuẩn hóa Min/Max', 'Chuẩn hóa Standard'],
    'Accuracy (Tập Test)': [acc_raw_lda, acc_minmax_lda, acc_std_lda]
})

display.display(results_lda)

# Vẽ biểu đồ trình diễn
plt.figure(figsize=(8, 5))
sns.barplot(x='Loại dữ liệu', y='Accuracy (Tập Test)', data=results_lda, palette='coolwarm')
plt.title('SO SÁNH HIỆU SUẤT LDA TRÊN CÁC LOẠI DỮ LIỆU', fontsize=14, fontweight='bold')
plt.ylabel('Độ chính xác (Accuracy)')
plt.ylim(0.60, 0.85)
for index, row in results_lda.iterrows():
    plt.text(index, row['Accuracy (Tập Test)'] + 0.005, f"{row['Accuracy (Tập Test)']:.4f}", color='black', ha="center")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
import joblib

print("--- 2. TINH CHỈNH THAM SỐ CHO LDA (TUNING) ---")

# 1. Định nghĩa các tham số cần tinh chỉnh cho LDA
# (Lưu ý: solver 'svd' không dùng được shrinkage, nên ta phải chia ra 2 bộ tham số)
param_grid_lda = [
    {'solver': ['svd']}, 
    {'solver': ['lsqr', 'eigen'], 'shrinkage': ['auto', 0.1, 0.5, 0.9]}
]

# 2. Khởi tạo và chạy dò tìm trên dữ liệu Standard
grid_search_lda = GridSearchCV(LinearDiscriminantAnalysis(), param_grid_lda, cv=10, scoring='accuracy')
grid_search_lda.fit(X_train_std, y_train)

# 3. Lấy ra mô hình xịn nhất
best_lda = grid_search_lda.best_estimator_
print("Tham số LDA tốt nhất tìm được:", grid_search_lda.best_params_)

# 4. Kiểm nghiệm mô hình tinh chỉnh trên tập Test
y_pred_tuned_lda = best_lda.predict(X_test_std)
acc_tuned_lda = accuracy_score(y_test, y_pred_tuned_lda)

print("Test Accuracy (Sau tinh chỉnh):", acc_tuned_lda)
cm_tuned = confusion_matrix(y_test, y_pred_tuned_lda)
print("Ma trận nhầm lẫn:\n", cm_tuned)

# 5. LƯU MÔ HÌNH TINH CHỈNH
joblib.dump(best_lda, 'lda_tuned_pima.pkl')

In [ ]:
import os
import pandas as pd
import joblib

print("--- 3. XUẤT KẾT QUẢ VÀ LƯU MÔ HÌNH ---")

# 1. Tạo bảng tóm tắt so sánh Baseline vs Tinh chỉnh
summary_df = pd.DataFrame({
    "Phiên bản Mô hình": ["Baseline (Tham số mặc định)", "Tuned (Đã tinh chỉnh)"],
    "Test Accuracy": [acc_std_lda, acc_tuned_lda], # Lấy điểm của data standard (Ô 1) đọ với điểm Tuned (Ô 2)
    "Bộ tham số": ["Mặc định", str(grid_search_lda.best_params_)]
})

# 2. Ma trận nhầm lẫn của mô hình Tinh chỉnh
cm_df = pd.DataFrame(cm_tuned)

# 3. K-Fold của mô hình Tinh chỉnh (nếu muốn lưu)
kfold_df = pd.DataFrame(grid_search_lda.cv_results_['mean_test_score'], columns=["KFold Mean Scores (All Params)"])

# ===== XUẤT FILE EXCEL =====
folder_path = "ket_qua"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

file_path = os.path.join(folder_path, "ket_qua_modelLinearDiscriminantAnalysis.xlsx")

with pd.ExcelWriter(file_path) as writer:
    # Sheet 1: So sánh 3 loại dữ liệu ban đầu
    results_lda.to_excel(writer, sheet_name="Baseline_3_Loai_Data", index=False)
    # Sheet 2: Trực tiếp đọ sức Baseline và Tuned
    summary_df.to_excel(writer, sheet_name="So_Sanh_Tuning", index=False)
    # Sheet 3: Ma trận nhầm lẫn cuối cùng
    cm_df.to_excel(writer, sheet_name="Confusion_Matrix_Tuned", index=False)
    # Sheet 4: Điểm K-fold
    kfold_df.to_excel(writer, sheet_name="KFold_Tuned", index=False)

print(f"Đã xuất báo cáo xuất sắc vào: {file_path}")

# ===== LƯU 2 MÔ HÌNH (Đúng chuẩn Mục 8 Đề bài) =====
joblib.dump(lda_eval, 'lda_baseline_pima.pkl') # Lưu mô hình gốc
joblib.dump(best_lda, 'lda_tuned_pima.pkl')    # Lưu mô hình đã độ tham số
print("Đã lưu cả 2 file mô hình (.pkl) thành công!")

In [ ]:
!jupyter nbconvert --to html Linear_Discrimination.ipynb